# MHS Formatting Notebook

This notebook formats **Measuring Hate Speech (MHS)** into the shared post-level schema used by the downstream union/dedup pipeline.

## What this notebook does
1. Loads MHS from local parquet (or from a Hugging Face parquet URI when requested).
2. Harmonizes annotator-level rows into the shared schema.
3. Aggregates annotator rows to one post-level row.
4. Saves standardized output to `outputs/preprocessing/02_mhs_standardized.tsv`.

## Output schema
- `post_id`
- `text`
- `raw_label`
- `binary_hate`
- `targets`
- `dataset`
- `text_dedup_key`
- `n_annotations`

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional
import json
import re

import pandas as pd

# Resolve a stable project root for notebook execution in multiple contexts.
WORKDIR = Path.cwd()
if not (WORKDIR / 'data').exists():
    WORKDIR = Path('/Users/RevaH/Documents/COS534/benchmarking_dogwhistles')

@dataclass
class MHSConfig:
    # Local/remote source configuration.
    mhs_local_path: Path = WORKDIR / 'data' / 'measuring_hate_speech.parquet'
    mhs_hf_uri: str = 'hf://datasets/ucberkeley-dlab/measuring-hate-speech/data/train-00000-of-00001.parquet'
    refresh_mhs_local_copy: bool = False

    # Threshold used to binarize the post-level mean hate speech score.
    mhs_primary_threshold: float = 0.5

    # Standardized output location.
    output_path: Path = WORKDIR / 'outputs' / 'preprocessing' / '02_mhs_standardized.tsv'

cfg = MHSConfig()
cfg.output_path.parent.mkdir(parents=True, exist_ok=True)
cfg.mhs_local_path.parent.mkdir(parents=True, exist_ok=True)
cfg

In [ ]:
def get_mhs_dataframe(local_path: Path, hf_uri: str, refresh_local: bool = False) -> pd.DataFrame:
    """Load MHS parquet from local cache or fetch-and-cache it from Hugging Face."""
    if local_path.exists() and not refresh_local:
        print(f'Loading MHS from local copy: {local_path}')
        return pd.read_parquet(local_path)

    print('Fetching MHS from Hugging Face URI...')
    df = pd.read_parquet(hf_uri)
    df.to_parquet(local_path, index=False)
    print(f'Saved local MHS copy to: {local_path}')
    return df


def normalize_text_for_dedup(text: str) -> str:
    """Create a lowercase, whitespace-normalized text key for dedup fallback."""
    if not isinstance(text, str):
        return ''
    return re.sub(r'\s+', ' ', text.strip().lower())


def is_truthy(value) -> bool:
    """Interpret booleans safely across bool/int/float/string representations."""
    if value is None:
        return False
    try:
        if pd.isna(value):
            return False
    except Exception:
        pass

    if isinstance(value, bool):
        return value
    if isinstance(value, (int, float)):
        return value != 0

    s = str(value).strip().lower()
    return s in {'1', 'true', 't', 'yes', 'y'}


def normalize_target_token(token: str) -> Optional[str]:
    """Normalize one target token to a stable lowercase form."""
    if token is None:
        return None
    value = str(token).strip().lower()
    if not value or value in {'none', 'null', 'nan'}:
        return None
    return value


def parse_targets_generic(value) -> List[str]:
    """Normalize mixed target payloads into a clean list[str]."""
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        candidates = list(value)
    elif isinstance(value, str):
        s = value.strip()
        if not s:
            return []
        if s.startswith('[') and s.endswith(']'):
            try:
                parsed = json.loads(s.replace("'", '"'))
                candidates = parsed if isinstance(parsed, list) else [parsed]
            except Exception:
                candidates = [s]
        elif '|' in s:
            candidates = [x.strip() for x in s.split('|')]
        elif ',' in s:
            candidates = [x.strip() for x in s.split(',')]
        else:
            candidates = [s]
    else:
        try:
            if pd.isna(value):
                return []
        except Exception:
            pass
        candidates = [value]

    normalized = []
    for candidate in candidates:
        token = normalize_target_token(candidate)
        if token:
            normalized.append(token)
    return sorted(set(normalized))


def serialize_targets_for_tsv(value) -> str:
    """Serialize list-like targets as pipe-delimited strings for TSV output."""
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ''
    if isinstance(value, list):
        return '|'.join(str(x) for x in value)
    return str(value)

In [ ]:
def extract_mhs_targets_from_row(row: pd.Series, target_cols: List[str]) -> List[str]:
    """Collapse MHS multi-hot target_* indicators into one normalized target list."""
    targets = []
    for col in target_cols:
        if not is_truthy(row.get(col)):
            continue

        suffix = col[len('target_'):]

        # Skip axis-only columns (e.g. target_race) — only keep axis_group columns (e.g. target_race_white)
        if '_' not in suffix:
            continue

        axis, group = suffix.split('_', 1)
        norm = normalize_target_token(f'{axis}_{group}')
        if norm:
            targets.append(norm)

    return sorted(set(targets))

In [ ]:
def harmonize_mhs(df: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Standardize MHS annotator-level rows into the common in-memory schema."""
    required = ['text', 'hate_speech_score']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise KeyError(f'MHS columns missing: {missing}')

    out = pd.DataFrame()
    out['post_id'] = df['comment_id'].astype(str) if 'comment_id' in df.columns else pd.Series(df.index).astype(str)
    out['text'] = df['text'].astype(str)
    out['raw_label'] = pd.to_numeric(df['hate_speech_score'], errors='coerce')
    out['binary_hate'] = (out['raw_label'] >= threshold).astype('Int64')

    target_cols = [col for col in df.columns if col.startswith('target_')]
    if target_cols:
        out['targets'] = df.apply(lambda row: extract_mhs_targets_from_row(row, target_cols), axis=1)
    elif 'target' in df.columns:
        out['targets'] = df['target'].apply(parse_targets_generic)
    else:
        out['targets'] = [[] for _ in range(len(df))]

    out['dataset'] = 'mhs'
    out['text_dedup_key'] = out['text'].apply(normalize_text_for_dedup)
    return out


def aggregate_mhs_to_post_level(mhs_harmonized: pd.DataFrame, threshold: float) -> pd.DataFrame:
    """Aggregate annotator-level rows to one row per post ID and text."""
    required = {'post_id', 'text', 'raw_label', 'targets', 'dataset', 'text_dedup_key'}
    missing = required - set(mhs_harmonized.columns)
    if missing:
        raise KeyError(f'MHS harmonized dataframe missing columns: {sorted(missing)}')

    base = mhs_harmonized.copy()
    base['raw_label'] = pd.to_numeric(base['raw_label'], errors='coerce')

    grouped = (
        base.groupby(['post_id', 'text', 'dataset', 'text_dedup_key'], dropna=False)
        .agg(
            raw_label=('raw_label', 'mean'),
            n_annotations=('raw_label', 'size'),
            targets=('targets', lambda s: sorted(set(t for v in s for t in (v if isinstance(v, list) else [])))),
        )
        .reset_index()
    )

    grouped['binary_hate'] = (grouped['raw_label'] >= threshold).astype('Int64')

    cols = ['post_id', 'text', 'raw_label', 'binary_hate', 'targets', 'dataset', 'text_dedup_key', 'n_annotations']
    return grouped[cols]


def dataframe_for_tsv(df: pd.DataFrame) -> pd.DataFrame:
    """Convert list columns into stable string-safe TSV representations."""
    export_df = df.copy()
    if 'targets' in export_df.columns:
        export_df['targets'] = export_df['targets'].apply(serialize_targets_for_tsv)
    return export_df

In [ ]:
mhs_raw = get_mhs_dataframe(cfg.mhs_local_path, cfg.mhs_hf_uri, refresh_local=cfg.refresh_mhs_local_copy)
mhs_harmonized = harmonize_mhs(mhs_raw, threshold=cfg.mhs_primary_threshold)
mhs_std = aggregate_mhs_to_post_level(mhs_harmonized, threshold=cfg.mhs_primary_threshold)
mhs_std_tsv = dataframe_for_tsv(mhs_std)

mhs_std_tsv.to_csv(cfg.output_path, sep='\t', index=False)

print('Saved MHS standardized output to:', cfg.output_path)
print('Rows written:', len(mhs_std_tsv))
display(mhs_std.head(3))

In [ ]:
# Compare target-label distributions side by side:
# 1) all rows
# 2) rows where binary_hate == 1
# Targets may be Python lists or list-like strings, so parse each value defensively.

import matplotlib.pyplot as plt
from IPython.display import HTML, display


def build_target_distribution(df: pd.DataFrame, targets_col: str = 'targets') -> pd.DataFrame:
    """Return count/percentage distribution from mixed target representations."""
    flat_labels = []

    for value in df[targets_col]:
        # parse_targets_generic handles list, tuple, set, plain strings, and list-like strings.
        labels = parse_targets_generic(value)
        if labels:
            flat_labels.extend(labels)

    if not flat_labels:
        return pd.DataFrame(columns=['target_label', 'count', 'percentage'])

    dist = pd.Series(flat_labels).value_counts().rename_axis('target_label').reset_index(name='count')
    dist['percentage'] = (dist['count'] / dist['count'].sum() * 100).round(2)
    return dist


# Use post-level dataframe directly so targets stay in their native list form.
work_df = mhs_std.copy()
work_df['binary_hate'] = pd.to_numeric(work_df['binary_hate'], errors='coerce').astype('Int64')

all_dist = build_target_distribution(work_df)
hate_only_df = work_df[work_df['binary_hate'] == 1].copy()
hate_dist = build_target_distribution(hate_only_df)

# Build side-by-side tables for direct comparison.
all_display = all_dist.copy()
all_display.columns = ['target_label', 'count_all', 'percentage_all']

hate_display = hate_dist.copy()
hate_display.columns = ['target_label', 'count_hate_only', 'percentage_hate_only']

comparison_table = (
    all_display
    .merge(hate_display, on='target_label', how='outer')
    .fillna(0)
)
comparison_table['count_all'] = comparison_table['count_all'].astype(int)
comparison_table['count_hate_only'] = comparison_table['count_hate_only'].astype(int)
comparison_table = comparison_table.sort_values(['count_all', 'count_hate_only'], ascending=False).reset_index(drop=True)

print('Side-by-side target label comparison table:')
display(comparison_table)

# Display separate tables side by side for readability.
all_html = all_dist.to_html(index=False)
hate_html = hate_dist.to_html(index=False)
display(HTML(
    '<div style="display:flex; gap:24px; align-items:flex-start;">'
    '<div style="flex:1;"><h4>All Rows</h4>' + all_html + '</div>'
    '<div style="flex:1;"><h4>binary_hate == 1</h4>' + hate_html + '</div>'
    '</div>'
))

# Side-by-side pie charts for quick composition comparison.
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

if not all_dist.empty:
    axes[0].pie(
        all_dist['count'],
        labels=all_dist['target_label'],
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.8,
    )
    axes[0].set_title('All Rows: Target Label Distribution')
else:
    axes[0].text(0.5, 0.5, 'No target labels', ha='center', va='center')
    axes[0].set_title('All Rows: Target Label Distribution')

if not hate_dist.empty:
    axes[1].pie(
        hate_dist['count'],
        labels=hate_dist['target_label'],
        autopct='%1.1f%%',
        startangle=140,
        pctdistance=0.8,
    )
    axes[1].set_title('binary_hate == 1: Target Label Distribution')
else:
    axes[1].text(0.5, 0.5, 'No target labels', ha='center', va='center')
    axes[1].set_title('binary_hate == 1: Target Label Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Change plot: compare target-label share in the original dataset (all rows)
# against the filtered subset (rows where binary_hate == 1).
# Positive values mean a label is more prevalent after filtering.
# Negative values mean a label is less prevalent after filtering.

# Merge distributions so each label has both baseline and filtered counts.
shift_df = all_dist.merge(
    hate_dist,
    on='target_label',
    suffixes=('_all', '_filtered'),
    how='outer'
).fillna(0)

# Use safe denominators to avoid divide-by-zero when a side has no labels.
all_total = shift_df['count_all'].sum()
filtered_total = shift_df['count_filtered'].sum()

shift_df['pct_all'] = (shift_df['count_all'] / all_total * 100) if all_total else 0.0
shift_df['pct_filtered'] = (shift_df['count_filtered'] / filtered_total * 100) if filtered_total else 0.0

# Delta is measured in percentage points: filtered minus original baseline.
shift_df['delta_pct_points'] = shift_df['pct_filtered'] - shift_df['pct_all']
shift_df = shift_df.sort_values('delta_pct_points')

# Color code: blue = increased share after filtering, gray = decreased share.
bar_colors = ['#0f90f2' if x > 0 else '#3b4046' for x in shift_df['delta_pct_points']]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(shift_df['target_label'], shift_df['delta_pct_points'], color=bar_colors)

# Zero line is the baseline where filtered share equals original share.
ax.axvline(0, color='black', linewidth=1.0)
ax.bar_label(bars, fmt='%+.1f pp', padding=4)

# Clear axis labels and title for interpretation.
ax.set_xlabel('Change in Share (percentage points): filtered - original')
ax.set_ylabel('Target label')
ax.set_title('Target Label Shift After Filtering to binary_hate == 1', pad=22)

# Subtitle clarifies directionality and comparison groups.
ax.text(
    0.5,
    1.02,
    'Baseline = all rows in HateXplain standardized output | Filtered = rows with binary_hate == 1',
    transform=ax.transAxes,
    ha='center',
    va='bottom',
    fontsize=10,
    color='#444444'
)

plt.tight_layout()
plt.show()

# Display the numeric shift table used to generate the chart.
display(shift_df[['target_label', 'count_all', 'pct_all', 'count_filtered', 'pct_filtered', 'delta_pct_points']])